# 노션 트러블슈팅 검색기


**프로젝트 개요**

기존 노션 검색의 한계인 제목 기반 검색을 벗어나, 문서 내용까지 의미적으로 검색할 수 있는 시스템을 구현했습니다.
```

노션 DB Export (Markdown)
        ↓
문서 청킹 (RecursiveCharacterTextSplitter)
        ↓
임베딩 변환 (OpenAI Embeddings)
        ↓
벡터 DB 저장 (Chroma)
        ↓
의미 기반 검색
        ↓
관련 문서 반환
```


In [1]:
import zipfile
import os

# 압축 풀기
with zipfile.ZipFile("data.zip", "r") as zip_ref:
    zip_ref.extractall("./notion_docs")

print("압축 해제 완료!")

# 파일 목록 확인
for root, dirs, files in os.walk("./notion_docs"):
    for file in files:
        print(file)

압축 해제 완료!
ExportBlock-f89a79cd-0b01-4d26-abdc-e7b047031d39-Part-1.zip


In [2]:
import zipfile
import os

# 압축 풀기
with zipfile.ZipFile("notion_docs/ExportBlock-f89a79cd-0b01-4d26-abdc-e7b047031d39-Part-1.zip", "r") as zip_ref:
    zip_ref.extractall("./notion_docs/data")

print("압축 해제 완료!")

# 파일 목록 확인
for root, dirs, files in os.walk("./notion_docs/data"):
    for file in files:
        print(file)

압축 해제 완료!
트러블슈팅 ae9e1903be104960aed1364937c6d431_all.csv
트러블슈팅 ae9e1903be104960aed1364937c6d431.csv
org hibernate proxy pojo bytebuddy ByteBuddyInterc 17daca7eedd180cda936fe4225ee4622.md
alter table 락 걸림 240aca7eedd180049eabef3c919a621b.md
springdoc - array 타입이 스웨거에서 이상하게 표시 198aca7eedd18065b211d0967839d244.md
java 17 이후 문법 차이 toLIst() vs collect(Collectors to 328aca7eedd18051819ff9278ed701b2.md
fedex internal 에러 배포 이후 ssl에러 지속- 1차 원인은 “HTTP 풀 O 318aca7eedd180609d89ffdfb6c1a8f9.md
오라로그 - 몽고 db e0abc847ba764d0cbdecb22e983000eb.md
[미해결]젠킨스 빌드 시 테스트 실패 원인 19aaca7eedd1801ea3c9f192a4a6f687.md
@RestControllerAdvice와 swagger 500에러 98a8c48a8f254af2a24468c66b376bd3.md
윈도우서버에서 아파치 실행안됨 47a709942e2e4e67a05e408dd4f9cb9d.md
운영 서버 배포 후 502 229aca7eedd18066b403f676a142409f.md
개발도메인 api 실행오류 - ajp 포트, 메모리 269aca7eedd18078b75fd245151c9bfa.md
select key 안됨 6b464f1cffed4a3aacd31ae19caab8bc.md
모듈별 common 설정 파일 분리 2342d67b7f68466587ac42d0a8efd02e.md
거래명세서 매출확정 오류 2fcaca7eedd18037be57fac18fc4634c.md
Mybatis

In [3]:
# 전체 파일 구조 확인 먼저!
import os

for root, dirs, files in os.walk("./notion_docs"):
    for file in files:
        if file.endswith(".md"):
            full_path = os.path.join(root, file)
            print(full_path)

./notion_docs/data/개인 페이지 & 공유된 페이지/트러블슈팅/org hibernate proxy pojo bytebuddy ByteBuddyInterc 17daca7eedd180cda936fe4225ee4622.md
./notion_docs/data/개인 페이지 & 공유된 페이지/트러블슈팅/alter table 락 걸림 240aca7eedd180049eabef3c919a621b.md
./notion_docs/data/개인 페이지 & 공유된 페이지/트러블슈팅/springdoc - array 타입이 스웨거에서 이상하게 표시 198aca7eedd18065b211d0967839d244.md
./notion_docs/data/개인 페이지 & 공유된 페이지/트러블슈팅/java 17 이후 문법 차이 toLIst() vs collect(Collectors to 328aca7eedd18051819ff9278ed701b2.md
./notion_docs/data/개인 페이지 & 공유된 페이지/트러블슈팅/fedex internal 에러 배포 이후 ssl에러 지속- 1차 원인은 “HTTP 풀 O 318aca7eedd180609d89ffdfb6c1a8f9.md
./notion_docs/data/개인 페이지 & 공유된 페이지/트러블슈팅/오라로그 - 몽고 db e0abc847ba764d0cbdecb22e983000eb.md
./notion_docs/data/개인 페이지 & 공유된 페이지/트러블슈팅/[미해결]젠킨스 빌드 시 테스트 실패 원인 19aaca7eedd1801ea3c9f192a4a6f687.md
./notion_docs/data/개인 페이지 & 공유된 페이지/트러블슈팅/@RestControllerAdvice와 swagger 500에러 98a8c48a8f254af2a24468c66b376bd3.md
./notion_docs/data/개인 페이지 & 공유된 페이지/트러블슈팅/윈도우서버에서 아파치 실행안됨 47a709942e2e4e67a05e408dd4f9cb9d.md
.

In [14]:
!pip install -qU langchain-chroma
!pip install -qU langchain-google-genai
!pip install -qU langchain-openai
!pip install langchain-text-splitters -q


In [26]:
from langchain_chroma import Chroma
from langchain_openai import OpenAIEmbeddings
from langchain_text_splitters import RecursiveCharacterTextSplitter  # 변경!
from dotenv import load_dotenv
import os

load_dotenv(dotenv_path="/content/.env", override=True)

# md 파일 전부 읽기
docs = []
file_names = []

for root, dirs, files in os.walk("./notion_docs"):
    for file in files:
        if file.endswith(".md"):
            full_path = os.path.join(root, file)
            with open(full_path, "r", encoding="utf-8") as f:
                content = f.read()
                if content.strip():  # 빈 파일 제외
                    docs.append(content)
                    file_names.append(file)

print(f"📄 총 {len(docs)}개 문서 로드!")
print("\n파일 목록:")
for name in file_names:
    print(f"  - {name}")

NameError: name 'Document' is not defined

In [29]:
from langchain_core.documents import Document


# 1. md 파일 읽기 (파일명 포함!)
docs = []
for root, dirs, files in os.walk("./notion_docs"):
    for file in files:
        if file.endswith(".md"):
            full_path = os.path.join(root, file)
            with open(full_path, "r", encoding="utf-8") as f:
                content = f.read()
                if content.strip():
                    docs.append(Document(
                        page_content=content,
                        metadata={"source": file}  # 파일명 저장!
                    ))

In [30]:
# 2. 청킹 (메타데이터 유지됨)
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=50
)
chunks = text_splitter.split_documents(docs)  # create_documents → split_documents!
print(f"✂️ 총 {len(chunks)}개 청크!")

# 3. 벡터 DB 저장
embeddings = OpenAIEmbeddings()
vector_store = Chroma.from_documents(
    documents=chunks,
    embedding=embeddings,
    collection_name="troubleshooting_db2",
    persist_directory="./notion_chroma2"
)
print(f"✅ 저장 완료! 총 {vector_store._collection.count()}개")

✂️ 총 662개 청크!
✅ 저장 완료! 총 662개


In [33]:
results = vector_store.similarity_search("jenkins", k=5)

print(f"검색 결과 {len(results)}개\n")
for i, doc in enumerate(results):
    print(f"=== {i+1}번 문서 ===")
    print(f"📄 제목: {doc.metadata['source']}")  # 파일명!
    print(f"📝 내용: {doc.page_content[:200]}")
    print()

검색 결과 5개

=== 1번 문서 ===
📄 제목: [PR]Administrator 계정 이름 변경 후 jenkins 자동 배포 실패 19aaca7eedd180fead22d109c00aac83.md
📝 내용: ![스크린샷 2025-02-14 오전 10.28.07.png](%5BPR%5DAdministrator%20%EA%B3%84%EC%A0%95%20%EC%9D%B4%EB%A6%84%20%EB%B3%80%EA%B2%BD%20%ED%9B%84%20jenkins%20%EC%9E%90%EB%8F%99%20%EB%B0%B0%ED%8F%AC%20%EC%8

=== 2번 문서 ===
📄 제목: [PR]Administrator 계정 이름 변경 후 jenkins 자동 배포 실패 19aaca7eedd180fead22d109c00aac83.md
📝 내용: ![스크린샷 2025-02-14 오전 11.06.55.png](%5BPR%5DAdministrator%20%EA%B3%84%EC%A0%95%20%EC%9D%B4%EB%A6%84%20%EB%B3%80%EA%B2%BD%20%ED%9B%84%20jenkins%20%EC%9E%90%EB%8F%99%20%EB%B0%B0%ED%8F%AC%20%EC%8

=== 3번 문서 ===
📄 제목: jenkins hook url 정리 304aca7eedd180ad887df0f53fc253d4.md
📝 내용: script {
                        try{
                 
                            // JAR 파일 실행
                            bat '''
                            title dentrion_tablet
                 

=== 4번 문서 ===
📄 제목: PR 서버에 적용하면서 문제 및 개선사항 정리 (feat dentrion_tablet_ap 19aaca7eedd180d